### This notebook compute "P9. Geology of the lagoon frontal zone" indicator for the 27 basins of IKI Project

**Created:** 6/23/2025 by Jorge Mayo (jmayo@rti.org)  / Serena Gilson
**Project #:** 0219481  
**Last modified:**
**Status:** Complete but elminated due to not enough information. 
**QA Status:** reviewed by  
**Original Script Stored at:** Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\Exposicion\Scripts_Exposure
 
**Objective:**   Calculate the geology of the glacial lagoons for each COMID
**Compatibility:** 
**Packages:** numpy, pandas, geopandas, sqlite3, matplotlib ...  
**Further documentation:**  
 
**Inputs:**   GEOCATMIN Mapa Geologico (Integrado_100k.gdb), MINAM MAPA NACIONAL DE ECOSISTEMAS 2012-2018 (selected out ECO_LAYER= "Lagos and Lagunas")
**Outputs:** 
 
**Assumptions:** Finding the majority geology for all the lagoon within the COMID (not attempting to locate the "frontal zone")
 
**Future work:** 
 
**Notes:** 

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt


In [2]:
#user = 'jmayo'
#user= 'cpickering'
user = 'sgilson'
ScnID= 1 #Scenario ID -- 1: Baseline, 1981-2005, 2: Future, 2030, 3: Future: 2050
IndID= 109 #Indicator ID (Exposure = 2 + 0X where X is the Exposure Indicator number, Peligro= 1 +0x, VSB= 3 +0x, VSS= 4 +0x, VCA= 5 +0x)
db_path = fr'C:\Users\{user}\Research Triangle Institute\IKI Peru Project - General\Interno\AI2a_Metodologia\Indicadores\BD_RiesgoClimatico_IKI.db'

In [ ]:
## FILEPATHS FOR MISSING SHAPEFILES

In [ ]:
#load and visualize geodatabase with attribute table
mapa_geo_gdf = gpd.read_file(mapa_geo, layer='geologia_100k')
#read shapefile
mapa_ecosistemas_gdf= gpd.read_file(mapa_ecosistemas)

#Filter mapa_ecosistemas_gdf for ECO_LAYER= "Lagos and Lagunas"
mapa_lag_gdf = mapa_ecosistemas_gdf[mapa_ecosistemas_gdf['ECO_LAYER'] == 'Lago y laguna']

In [ ]:
#list of all unique values in DESCRIP column
#mapa_geo_gdf
#force the list of all values to appear
UNIDAD_unique= mapa_geo_gdf['UNIDAD'].unique()
UNIDAD_unique_gdf= pd.DataFrame(UNIDAD_unique, columns=['UNIDAD'])

In [ ]:
#according to the geology type, assign a score: 

#ASSIGN SCORES
#looking into how to associate geology types with descriptors here
#note will likely need to add accents here
#save score as a column in geodataframe
#need to iterate through each feature/ensure the score is being saved for each feature
# Define a mapping of geology types to scores
geology_scores = {
    'macizo rocoso intrusivo': 0,
    'macizo rocoso sedimentario/metamorfico': 0.25,
    'deposito cuaternario/macizo rocoso': 0.5,
    'deposito cuaternario glaciarico': 0.75,
    'otro tipo de deposito cuaternario': 1
}

# Ensure the 'GEOL' column exists and contains valid data
if 'GEOL' in mapa_lag_gdf.columns:
    # Assign scores based on the geology type
    mapa_lag_gdf['geology_score'] = mapa_lag_gdf['GEOL'].map(geology_scores)

    # Handle cases where the geology type is not in the mapping
    mapa_lag_gdf['geology_score'] = mapa_lag_gdf['geology_score'].fillna(0)  # Default score for unknown types

    # Group by each lagoon or feature and find the majority geology type
    majority_geology = mapa_lag_gdf.groupby('feature_id')['GEOL'].agg(lambda x: x.mode()[0])

    # Assign the majority geology score back to the GeoDataFrame
    mapa_lag_gdf['majority_geology_score'] = mapa_lag_gdf['feature_id'].map(
        majority_geology.map(geology_scores)
    )
else:
    print("The 'GEOL' column is missing in the GeoDataFrame.")

#then, for each subbasin, find the median geology score of all the lagoons within that subbasin
zonal_stats_results = zonal_stats(subbasins_gdf, mapa_lag_gdf['majority_geology_score'], stats=['median'], geojson_out=True)
#add median geology score to subbasins_gdf
subbasins_gdf['median'] = [feature['properties']['median'] for feature in zonal_stats_results]
#view subbasins_gdf with median geology score
subbasins_gdf[['COMID', 'median']]

#plot the score for the subbasins
subbasins_gdf.plot(column='median', cmap='OrRd', legend=True)


SQLITE INSERT 

In [ ]:
#Update relevant dataframe and value column from the calculations above for the specific indicator
insert_data= subbasins_gdf #COMID 
value_column= 'median'

In [ ]:
# Connect to your SQLite database
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Fixed IndID and ScnID (update above)

#For now I am commenting out the looping of scenarios as I imagine we might run just baseline scenarios for now.
# Define the mapping between scenario and DataFrame column
# scenario_columns = {
#     1: 'ACTUAL ANUAL',
#     2: '2030 ANUAL',
#     3: '2050 ANUAL'
# }

# Prepare list of rows to insert
rows_to_insert = []

#for scn_id, column_name in scenario_columns.items():
for _, row in insert_data.iterrows():
    comid = row['COMID']
    value = row[value_column]
    #normalized_value = None  # or compute something like: value / max_value #took out normalized value from the table for now
    rows_to_insert.append((ScnID, IndID, comid, value))

# Insert data into IndValues_Dyn
insert_query = """
INSERT OR REPLACE INTO IndValues_Dyn (ScnID, IndID, COMID, Value)
VALUES (?, ?, ?, ?);
"""


In [ ]:
rows_to_insert

[(1, 106, 311153600, 3.181939799331104),
 (1, 106, 310832400, 2.5108010801080107),
 (1, 106, 310825700, 2.511406844106464),
 (1, 106, 310282400, 2.0313291139240506),
 (1, 106, 308754600, 3.530734106076572),
 (1, 106, 307736500, 1.2446808510638299),
 (1, 106, 307078200, 4.03176085941149),
 (1, 106, 307183500, 4.283126293995859),
 (1, 106, 306538400, 1.5944931163954943),
 (1, 106, 306599700, 1.2647535292756307),
 (1, 106, 306319300, 2.0713717693836977),
 (1, 106, 306030000, 2.4648941662364483),
 (1, 106, 305746400, 1.0184089414858646),
 (1, 106, 311953100, 2.0),
 (1, 106, 304273900, 1.0),
 (1, 106, 304281100, 1.0),
 (1, 106, 304281200, 1.0),
 (1, 106, 304379300, 1.0),
 (1, 106, 304274000, 1.0),
 (1, 106, 304313700, 1.0),
 (1, 106, 304306400, 1.0),
 (1, 106, 304327700, 1.0),
 (1, 106, 304313800, 1.0),
 (1, 106, 304306300, 1.0),
 (1, 106, 304321600, 1.0),
 (1, 106, 304321700, 1.0005494505494505),
 (1, 106, 304339400, 1.0),
 (1, 106, 304335700, 1.0615627466456197),
 (1, 106, 304343200, 1.0)

In [ ]:
# Check for duplicates in the input dataframe before insert
df_check = pd.DataFrame(rows_to_insert, columns=['ScnID', 'IndID', 'COMID', 'Value'])
duplicates = df_check.duplicated(subset=['ScnID', 'IndID', 'COMID'])
print("Duplicates in rows_to_insert:", df_check[duplicates])

Duplicates in rows_to_insert: Empty DataFrame
Columns: [ScnID, IndID, COMID, Value]
Index: []


In [ ]:
#Execute insert to SQLite Database
cursor.executemany(insert_query, rows_to_insert)
conn.commit()
conn.close()